In [2]:
# Combine notebook
#
# Reads the base grid + static layers + the CDR snapshot, applies the combine math from
# the grid-layer architecture plan, and writes data/rate-grid.json: the single combined
# grid the browser samples at runtime (app/globe/useGlobeData.js's sampleCell()).
#
#   r(m49) = CDR(m49) * WBpop(m49) / (1000 * gridPop(m49))   -- per-gridded-person rate
#   r_bar  = sum(deaths) / sum(gridPop)                       -- population-weighted global mean
#   w(cell) = cellPop * r_bar * country-rate(cell)
#
# By construction: sum(w) == global deaths/year, and per-country sum(w) == CDR*WBpop/1000
# exactly -- this reproduces today's country-level Poisson rates, not an approximation.
#
# Coverage: 3 countries (Montenegro 499, Serbia 688, South Sudan 728) have a CDR but no
# base-grid cells (GPWv4's border-cell assignment gave all their raster bins to a
# neighbor). scripts/gen-synthetic-cells.mjs pre-generated cells for them
# (data/source/synthetic-cells.json); this notebook folds those into the base grid before
# baking so they don't silently vanish.

import sys
sys.path.insert(0, "lib")
import grid

base_cells, cellsize = grid.load_base()
print(f"{len(base_cells)} base-grid cells, cellsize={cellsize}")


59863 base-grid cells, cellsize=0.5


In [3]:
import json

synthetic = json.loads((grid.DATA_DIR / "source" / "synthetic-cells.json").read_text())
assert synthetic["cellsize"] == cellsize, "synthetic cells must share the base grid's cellsize"

synthetic_cells = [
    {"lon": lon, "lat": lat, "pop": pop, "m49": m49}
    for lon, lat, pop, m49 in synthetic["cells"]
]
all_cells = base_cells + synthetic_cells
print(f"{len(synthetic_cells)} synthetic cells added -> {len(all_cells)} total cells.")
print("Countries added:", sorted({c['m49'] for c in synthetic_cells}))


91 synthetic cells added -> 59954 total cells.
Countries added: [499, 688, 728]


In [5]:
cdr = grid.load_cdr_snapshot()
country_rate_layer = grid.load_layer("country-rate")
assert country_rate_layer["dynamic"] is False
print(f"{len(cdr)} CDR countries, country-rate layer has {len(country_rate_layer['cells'])} cells.")

# r(m49) and r_bar, recomputed here over ALL cells (base + synthetic) so the 3 synthetic
# countries participate on equal footing. Recomputing (rather than only reading the
# pre-baked layer file) also sidesteps a subtle correctness trap: the layer file's values
# are mean-1 relative to the r_bar used when IT was baked (base cells only); reusing a
# different r_bar here without accounting for that would silently bias every country's
# rate by a small constant factor. Computing r(m49)/r_bar fresh, from the same cdr +
# gridPop inputs, guarantees w = cellPop * r_bar * (r/r_bar) = cellPop * r exactly,
# independent of any r_bar mismatch. The pre-baked layer is still loaded and checked
# below for the base cells, as a drift check between country-rate.ipynb and this notebook.
grid_pop_by_country = {}
for c in all_cells:
    grid_pop_by_country[c["m49"]] = grid_pop_by_country.get(c["m49"], 0) + c["pop"]

rate_by_country = {}
total_deaths = 0.0
total_grid_pop_with_cdr = 0.0
for m49, gpop in grid_pop_by_country.items():
    c = cdr.get(m49)
    if not c or not (gpop > 0):
        continue
    deaths = c["cdr"] * c["population"] / 1000.0
    rate_by_country[m49] = deaths / gpop
    total_deaths += deaths
    total_grid_pop_with_cdr += gpop

r_bar = total_deaths / total_grid_pop_with_cdr
print(f"Global deaths/year: {total_deaths:,.0f}")
print(f"r_bar: {r_bar:.6f}")

# Drift check: for the 59863 base cells, (rate_by_country[m49]/r_bar) should match the
# pre-baked country-rate.json value closely (small diff expected: this r_bar includes the
# 3 synthetic countries' population, country-rate.ipynb's did not).
country_rate_by_pos = {(lon, lat): v for lon, lat, m49, v in country_rate_layer["cells"]}
max_diff = 0.0
for c in base_cells:
    m49 = c["m49"]
    r = rate_by_country.get(m49)
    expected = r / r_bar if r is not None else 1.0
    actual = country_rate_by_pos[(c["lon"], c["lat"])]
    max_diff = max(max_diff, abs(expected - actual))
print(f"Max relative drift vs country-rate.json (expected small, from synthetic pop share): {max_diff:.4%}")


169 CDR countries, country-rate layer has 59863 cells.
Global deaths/year: 61,615,939
r_bar: 0.008452
Max relative drift vs country-rate.json (expected small, from synthetic pop share): 0.2124%


In [7]:
# Build data/rate-grid.json: w(cell) = cellPop * rate_by_country[m49] for every cell with
# a usable CDR; 0 for cells whose country has no CDR (Singapore, Hong Kong, ... -- never
# fire, matching today). names{} is embedded so the runtime doesn't need /api/mortality.

cdr_snapshot_raw = json.loads((grid.DATA_DIR / "source" / "cdr-snapshot.json").read_text())
names = {str(m49): c["name"] for m49, c in cdr.items()}

out_cells = []
total_w = 0.0
for c in all_cells:
    r = rate_by_country.get(c["m49"])
    w = c["pop"] * r if r is not None else 0.0
    out_cells.append([c["lon"], c["lat"], c["m49"], round(w, 6)])
    total_w += w

rate_grid = {
    "meta": {
        "year": cdr_snapshot_raw["year"],
        "sources": ["World Bank CDR + population (data/source/cdr-snapshot.json)",
                     "GPWv4 2015 population density (data/density-grid.json)",
                     "synthetic cells for Montenegro/Serbia/South Sudan (data/source/synthetic-cells.json)"],
        "baseRatePerPersonYear": r_bar,
        "totalDeathsPerYear": total_w,
    },
    "names": names,
    "cellsize": cellsize,
    "cells": out_cells,
}
print(f"{len(out_cells)} cells, total deaths/year = {total_w:,.0f}")


59954 cells, total deaths/year = 61,615,939


In [8]:
# Verification (plan §Verification #1):
# - grid m49 superset of CDR m49 (no country with a CDR silently vanishes)
# - per-country sum(w) matches CDR*WBpop/1000 exactly for a few large countries
# - global sum(w) matches sum of all countries' CDR*WBpop/1000

grid_m49 = {c[2] for c in out_cells if c[3] > 0}
cdr_m49 = set(rate_by_country.keys())
missing = cdr_m49 - grid_m49
assert not missing, f"CDR countries missing from the combined grid: {missing}"
print(f"OK: all {len(cdr_m49)} CDR-covered countries have grid weight.")

def country_total(m49):
    return sum(w for lon, lat, mid, w in out_cells if mid == m49)

for m49, label in [(840, "US"), (356, "India"), (276, "Germany"),
                    (688, "Serbia"), (499, "Montenegro"), (728, "South Sudan")]:
    expected = cdr[m49]["cdr"] * cdr[m49]["population"] / 1000.0
    actual = country_total(m49)
    diff = abs(actual - expected) / expected
    print(f"{label:12s} expected={expected:>14,.0f}  actual={actual:>14,.0f}  diff={diff:.6%}")
    assert diff < 1e-6, f"{label} deaths/year mismatch: {actual} vs {expected}"

assert abs(total_w - total_deaths) / total_deaths < 1e-9
print("OK: global total matches sum(CDR*WBpop/1000) across all countries.")


OK: all 169 CDR-covered countries have grid weight.
US           expected=     3,076,064  actual=     3,076,064  diff=0.000000%
India        expected=     9,642,482  actual=     9,642,482  diff=0.000000%
Germany      expected=     1,010,244  actual=     1,010,244  diff=0.000000%
Serbia       expected=        97,582  actual=        97,582  diff=0.000000%
Montenegro   expected=         6,356  actual=         6,356  diff=0.000000%
South Sudan  expected=       119,706  actual=       119,706  diff=0.000000%
OK: global total matches sum(CDR*WBpop/1000) across all countries.


In [9]:
out_path = grid.DATA_DIR / "rate-grid.json"
out_path.write_text(json.dumps(rate_grid))
print(f"Wrote {out_path.relative_to(grid.ROOT)}: {len(rate_grid['cells'])} cells, "
      f"{len(rate_grid['names'])} named countries, "
      f"{rate_grid['meta']['totalDeathsPerYear']:,.0f} deaths/year.")


Wrote data/rate-grid.json: 59954 cells, 169 named countries, 61,615,939 deaths/year.


New Notebook Created by Jupyter MCP Server